##Import Necessary Libraries

In [ ]:
!pip install tensorflow
!pip install scikit-learn==1.5.2
!pip install scikeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 884.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 26.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.21.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


  Using cached scikeras-0.13.0-py3-none-any.whl.metadata (3.1 kB)
Using cached scikeras-0.13.0-py3-none-any.whl (26 kB)


In [ ]:
!pip install scikeras

In [ ]:
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from scikeras.wrappers import KerasClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,GridSearchCV,KFold
import warnings
warnings.filterwarnings('ignore')

###Load Data

In [ ]:
diabetes_data = pd.read_csv('diabetes.csv')
diabetes_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


##Data Preparation

In [ ]:
X = diabetes_data.drop(columns='Outcome')
X

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [ ]:
y = diabetes_data['Outcome']
y

,Outcome
0,1
1,0
2,1
3,0
4,1
...,...
763,0
764,0
765,0
766,1


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=13,shuffle=True,stratify=y)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

##Model Building

In [ ]:
def create_model(learning_rate=0.001,dropout_rate=0.0,activation_function='relu',init='glorot_uniform',neuron1=8,neuron2=4):

  model = Sequential([

    keras.layers.Input(shape=(8,)),
    Dense(neuron1, kernel_initializer=init, activation=activation_function),
    Dropout(dropout_rate),
    Dense(neuron2, kernel_initializer=init, activation=activation_function),
    Dropout(dropout_rate),
    Dense(1,activation='sigmoid')
  ])
  optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
model = KerasClassifier(model=create_model,verbose=1)

In [ ]:
params_grid = {
    'model__learning_rate': [0.001,0.01],
    'model__dropout_rate': [0.0, 0.2],
    'model__activation_function': ['tanh','relu'],
    'model__init': ['glorot_uniform','he_uniform'],
    'model__neuron1': [8,16],
    'model__neuron2': [4,8],
    'batch_size':[10,20],
    'epochs':[10,50]
}

In [ ]:
kfold = KFold(n_splits=3,shuffle=True,random_state=45)
grid = GridSearchCV(estimator=model,param_grid=params_grid,cv=kfold,verbose=10,n_jobs=-1)


In [ ]:
grid_result = grid.fit(X_train,y_train)

Fitting 3 folds for each of 256 candidates, totalling 768 fits
Epoch 1/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4837 - loss: 0.8264
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5277 - loss: 0.7471
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5635 - loss: 0.6999
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5814 - loss: 0.6544
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6612 - loss: 0.6229
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6547 - loss: 0.6388
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6824 - loss: 0.5842
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6889 - loss: 0.5876
Epoch 9/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6678 - loss: 0.6026
Epoch 10/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7003 - loss: 0.5693
Epoch 11/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6971 - loss: 0.5786
Epoch 12/50

In [ ]:
print(f'Best:{grid_result.best_score_:.6f} using {grid_result.best_params_}')
means = grid_result.cv_results_['mean_test_score']
stds =  grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean,std,param in zip(means,stds,params):
  print(f'{mean:.6f} ({std:.6f} with: {param}')

Best:0.775251 using {'batch_size': 10, 'epochs': 50, 'model__activation_function': 'tanh', 'model__dropout_rate': 0.2, 'model__init': 'he_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 4}
0.755715 (0.007502 with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'tanh', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 4})
0.762259 (0.021505 with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'tanh', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 8})
0.754097 (0.016219 with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'tanh', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 16, 'model__neuron2': 4})
0.771991 (0.008162 with: {'batch_size': 10, 'epochs': 10, 'model__activation_function

In [ ]:
results_df = pd.DataFrame(grid_result.cv_results_)
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_batch_size,param_epochs,param_model__activation_function,param_model__dropout_rate,param_model__init,param_model__learning_rate,param_model__neuron1,param_model__neuron2,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,7.904274,2.995943,0.374932,0.044022,10,10,tanh,0.0,glorot_uniform,0.001,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.756098,0.746341,0.764706,0.755715,0.007502,81
1,4.165514,0.666762,0.423844,0.126641,10,10,tanh,0.0,glorot_uniform,0.001,8,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.760976,0.736585,0.789216,0.762259,0.021505,28
2,4.779909,0.757418,0.338984,0.057776,10,10,tanh,0.0,glorot_uniform,0.001,16,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.760976,0.731707,0.769608,0.754097,0.016219,90
3,4.945077,0.555143,0.470299,0.073361,10,10,tanh,0.0,glorot_uniform,0.001,16,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.780488,0.760976,0.774510,0.771991,0.008162,4
4,4.214599,0.475211,0.344527,0.075034,10,10,tanh,0.0,glorot_uniform,0.010,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.746341,0.741463,0.774510,0.754105,0.014565,87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,10.672616,0.321428,0.292055,0.074788,20,50,relu,0.2,he_uniform,0.001,16,8,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.765854,0.707317,0.754902,0.742691,0.025410,163
252,10.686885,0.274966,0.357356,0.092738,20,50,relu,0.2,he_uniform,0.010,8,4,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.780488,0.712195,0.750000,0.747561,0.027934,141
253,9.633203,0.334156,0.428022,0.053811,20,50,relu,0.2,he_uniform,0.010,8,8,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.770732,0.736585,0.759804,0.755707,0.014238,82
254,10.730132,0.060691,0.293194,0.068520,20,50,relu,0.2,he_uniform,0.010,16,4,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.726829,0.692683,0.769608,0.729707,0.031470,190


In [ ]:
results_df.to_csv('results.csv')

In [ ]:
import sklearn
import tensorflow as tf
import keras
import scikeras

print("scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("SciKeras:", scikeras.__version__)

scikit-learn: 1.6.1
TensorFlow: 2.20.0
Keras: 3.13.2
SciKeras: 0.13.0


In [ ]:
print(model)
from sklearn.base import is_classifier

print(is_classifier(model))